# 402 — Cross-Lineage Robustness

## Objective

Assess the cross-lineage robustness of the three frozen candidate cross-system
transcriptomic consensus representations constructed in notebook 401.

The analysis will determine whether each consensus representation retains
interpretable within-lineage variation and source-program concordance across
multiple tumor projects and cell-line lineages, while identifying cases in
which the global representation may be disproportionately influenced by a
restricted lineage context.

The consensus program identities, 2,389-gene weight vectors, orientations,
source-program mappings, tumor-arm context, and system-specific consensus scores
are frozen inputs from notebook 401. Notebook 402 does not redefine consensus
eligibility, correspondence classes, gene weights, orientations, source-program
identities, or tumor methylation arms.

## Analytical status

This notebook is a confirmatory internal-robustness analysis of the frozen
Phase 4 consensus layer.

Lineage-aware metrics, minimum group-size requirements, leave-one-lineage-out
procedures, resampling procedures, and any multiplicity rules that affect
interpretation must be specified before inspecting lineage-specific results.
Observed results will not be used to redefine these analytical rules.

## Scope and methodological boundary

Tumor and cell-line systems are evaluated separately. No joint cross-system
normalization, naïve pan-cancer pooling, or random pan-cancer splitting is
introduced.

Lineage is treated as a potentially important source of biological structure
and confounding rather than as variation to be removed automatically.

Pharmacogenomic phenotype, downstream biological annotation, epigenetic
regulator enrichment, scientific priority, and tumor-side methylation
information do not determine or rescue cross-lineage robustness. Frozen
upstream robustness and confounding annotations remain contextual evidence only.

Cross-lineage robustness in this notebook constitutes internal robustness of
the frozen consensus representation. It does not establish cross-dataset
replication, orthogonal support, biological causality, clinical prediction,
or full epigenetic-transcriptomic reproduction.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import pandas as pd

from scipy.stats import pearsonr, spearmanr

from pancancer_epigenetics.utils.paths import Paths

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

CONSENSUS_PROGRAM_DIR = Paths.consensus_programs
TUMOR_PROGRAM_DIR = Paths.tumor_programs
CELL_LINE_PROGRAM_DIR = Paths.cellline_programs
OUTPUT_DIR = Paths.consensus_programs

In [3]:
# =============================================================================
# Authoritative cross-lineage robustness input paths
# =============================================================================

CONSENSUS_CATALOG_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_program_catalog.csv"
)

TUMOR_CONSENSUS_SCORES_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_tumor_scores.parquet"
)

CELL_LINE_CONSENSUS_SCORES_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_cellline_scores.parquet"
)

TUMOR_NATIVE_SCORES_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_primary_tumor_rna_ica_candidate_scores.csv"
)

CELL_LINE_NATIVE_SCORES_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_scores.parquet"
)

CELL_LINE_COHORT_PATH = (
    Paths.metadata
    / "302_integrated_modeling_cohort.csv"
)

In [4]:
# =============================================================================
# Load authoritative cross-lineage robustness inputs
# =============================================================================

consensus_catalog = pd.read_csv(CONSENSUS_CATALOG_PATH)

tumor_consensus_scores = pd.read_parquet(
    TUMOR_CONSENSUS_SCORES_PATH
)

cell_line_consensus_scores = pd.read_parquet(
    CELL_LINE_CONSENSUS_SCORES_PATH
)

tumor_native_scores = pd.read_csv(
    TUMOR_NATIVE_SCORES_PATH
)

cell_line_native_scores = pd.read_parquet(
    CELL_LINE_NATIVE_SCORES_PATH
)

cell_line_cohort = pd.read_csv(
    CELL_LINE_COHORT_PATH
)


In [5]:
# =============================================================================
# Freeze cross-lineage robustness analysis parameters
# =============================================================================

MIN_LINEAGE_N = 15

# Lineage structure uses eta-squared across all groups. Lineage-specific
# fidelity uses Pearson and Spearman only for groups with n >= MIN_LINEAGE_N.
# LOLO evaluates every group without refitting or reweighting. No resampling,
# p-value gates, or categorical robustness threshold are introduced.

In [6]:
# =============================================================================
# Freeze consensus-program analysis set
# =============================================================================

consensus_programs = (
    consensus_catalog[
        [
            "consensus_program_id",
            "tumor_rna_axis",
            "cell_line_program",
            "orientation_multiplier",
        ]
    ]
    .copy()
)

consensus_program_ids = consensus_programs[
    "consensus_program_id"
].tolist()

consensus_programs

,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_09,1
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_29,-1
2,CONSENSUS_TX_03,RNA_IC184,ICA_PROGRAM_13,-1


In [7]:
# =============================================================================
# Construct lineage-aware consensus analysis tables
# =============================================================================

tumor_analysis = (
    tumor_consensus_scores
    .merge(
        tumor_native_scores[
            [
                "case_submitter_id",
                *consensus_programs["tumor_rna_axis"],
            ]
        ],
        on="case_submitter_id",
        how="inner",
        validate="one_to_one",
    )
)

cell_line_analysis = (
    cell_line_consensus_scores
    .merge(
        cell_line_cohort[
            [
                "ModelID",
                "OncotreeLineage",
            ]
        ],
        on="ModelID",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        cell_line_native_scores[
            [
                "ModelID",
                *consensus_programs["cell_line_program"],
            ]
        ],
        on="ModelID",
        how="inner",
        validate="one_to_one",
    )
)

print("Tumor analysis    :", tumor_analysis.shape)
print("Cell-line analysis:", cell_line_analysis.shape)

Tumor analysis    : (9965, 11)
Cell-line analysis: (713, 8)


In [8]:
# =============================================================================
# Define lineage groups eligible for within-lineage analyses
# =============================================================================

tumor_project_counts = tumor_analysis["project_id"].value_counts()

cell_line_lineage_counts = (
    cell_line_analysis["OncotreeLineage"].value_counts()
)

eligible_tumor_projects = tumor_project_counts[
    tumor_project_counts >= MIN_LINEAGE_N
].index.tolist()

eligible_cell_line_lineages = cell_line_lineage_counts[
    cell_line_lineage_counts >= MIN_LINEAGE_N
].index.tolist()

lineage_coverage_summary = pd.DataFrame(
    [
        {
            "system": "tumor",
            "total_groups": len(tumor_project_counts),
            "eligible_groups": len(eligible_tumor_projects),
            "eligible_samples": int(
                tumor_project_counts.loc[eligible_tumor_projects].sum()
            ),
        },
        {
            "system": "cell_line",
            "total_groups": len(cell_line_lineage_counts),
            "eligible_groups": len(eligible_cell_line_lineages),
            "eligible_samples": int(
                cell_line_lineage_counts.loc[
                    eligible_cell_line_lineages
                ].sum()
            ),
        },
    ]
)

lineage_coverage_summary

,system,total_groups,eligible_groups,eligible_samples
0,tumor,33,33,9965
1,cell_line,27,16,643


In [9]:
# =============================================================================
# Define lineage-variance effect-size helper
# =============================================================================

def eta_squared_by_group(data, value_column, group_column):
    """Quantify the fraction of score variance attributable to group structure."""

    values = data[value_column]
    grand_mean = values.mean()

    ss_total = ((values - grand_mean) ** 2).sum()

    group_summary = (
        data
        .groupby(group_column)[value_column]
        .agg(["size", "mean"])
    )

    ss_between = (
        group_summary["size"]
        * (group_summary["mean"] - grand_mean) ** 2
    ).sum()

    return ss_between / ss_total

In [10]:
# =============================================================================
# Quantify lineage structure of frozen consensus scores
# =============================================================================

lineage_eta_squared = []

for consensus_id in consensus_program_ids:
    lineage_eta_squared.extend(
        [
            {
                "system": "tumor",
                "consensus_program_id": consensus_id,
                "lineage_eta_squared": eta_squared_by_group(
                    tumor_analysis,
                    consensus_id,
                    "project_id",
                ),
            },
            {
                "system": "cell_line",
                "consensus_program_id": consensus_id,
                "lineage_eta_squared": eta_squared_by_group(
                    cell_line_analysis,
                    consensus_id,
                    "OncotreeLineage",
                ),
            },
        ]
    )

lineage_eta_squared = pd.DataFrame(lineage_eta_squared)

lineage_eta_squared

,system,consensus_program_id,lineage_eta_squared
0,tumor,CONSENSUS_TX_01,0.458909
1,cell_line,CONSENSUS_TX_01,0.894808
2,tumor,CONSENSUS_TX_02,0.809194
3,cell_line,CONSENSUS_TX_02,0.565508
4,tumor,CONSENSUS_TX_03,0.521870
5,cell_line,CONSENSUS_TX_03,0.612890


In [11]:
# =============================================================================
# Define within-lineage score summary helper
# =============================================================================

def summarize_within_lineage(
    data,
    value_column,
    group_column,
    eligible_groups,
):
    """Summarize consensus-score variation within prespecified eligible groups."""

    subset = data.loc[
        data[group_column].isin(eligible_groups),
        [group_column, value_column],
    ]

    summary = (
        subset
        .groupby(group_column)[value_column]
        .agg(
            n="size",
            mean="mean",
            median="median",
            sd="std",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    summary["iqr"] = summary["q75"] - summary["q25"]

    return summary

In [12]:
# =============================================================================
# Summarize within-lineage consensus-score variation
# =============================================================================

within_lineage_summaries = []

for consensus_id in consensus_program_ids:
    tumor_summary = summarize_within_lineage(
        tumor_analysis,
        consensus_id,
        "project_id",
        eligible_tumor_projects,
    ).assign(
        system="tumor",
        consensus_program_id=consensus_id,
    )

    cell_line_summary = summarize_within_lineage(
        cell_line_analysis,
        consensus_id,
        "OncotreeLineage",
        eligible_cell_line_lineages,
    ).assign(
        system="cell_line",
        consensus_program_id=consensus_id,
    )

    within_lineage_summaries.extend(
        [tumor_summary, cell_line_summary]
    )

within_lineage_summary = pd.concat(
    within_lineage_summaries,
    ignore_index=True,
)


In [13]:
# =============================================================================
# Harmonize lineage-group identifier
# =============================================================================

within_lineage_summary["lineage_group"] = (
    within_lineage_summary["project_id"]
    .combine_first(within_lineage_summary["OncotreeLineage"])
)

within_lineage_summary = (
    within_lineage_summary
    .drop(columns=["project_id", "OncotreeLineage"])
    .loc[
        :,
        [
            "system",
            "consensus_program_id",
            "lineage_group",
            "n",
            "mean",
            "median",
            "sd",
            "q25",
            "q75",
            "iqr",
        ],
    ]
)

within_lineage_summary

,system,consensus_program_id,lineage_group,n,mean,median,sd,q25,q75,iqr
0,tumor,CONSENSUS_TX_01,TCGA-ACC,79,-0.530955,-0.613182,0.636493,-1.058438,-0.216439,0.841998
1,tumor,CONSENSUS_TX_01,TCGA-BLCA,405,-0.450177,-0.596383,0.877973,-1.124293,0.051755,1.176048
2,tumor,CONSENSUS_TX_01,TCGA-BRCA,1089,-0.067551,-0.175151,0.771523,-0.599317,0.355618,0.954935
3,tumor,CONSENSUS_TX_01,TCGA-CESC,304,-0.051596,-0.078050,0.849842,-0.645156,0.543437,1.188593
4,tumor,CONSENSUS_TX_01,TCGA-CHOL,35,-0.270192,-0.472364,1.048007,-0.820140,0.049550,0.869690
...,...,...,...,...,...,...,...,...,...,...
142,cell_line,CONSENSUS_TX_03,Ovary/Fallopian Tube,34,0.183282,0.114965,0.542234,-0.218259,0.476715,0.694974
143,cell_line,CONSENSUS_TX_03,Pancreas,28,-0.519407,-0.602762,0.401465,-0.776597,-0.335383,0.441213
144,cell_line,CONSENSUS_TX_03,Peripheral Nervous System,18,0.955637,0.589712,0.727261,0.491863,1.002098,0.510234
145,cell_line,CONSENSUS_TX_03,Skin,37,0.854126,0.659436,0.626122,0.485200,1.181526,0.696325


In [14]:
# =============================================================================
# Summarize within-lineage dispersion across eligible groups
# =============================================================================

within_lineage_dispersion = (
    within_lineage_summary
    .groupby(
        [
            "system",
            "consensus_program_id",
        ]
    )
    .agg(
        eligible_groups=("lineage_group", "size"),
        median_sd=("sd", "median"),
        minimum_sd=("sd", "min"),
        maximum_sd=("sd", "max"),
        median_iqr=("iqr", "median"),
        minimum_iqr=("iqr", "min"),
        maximum_iqr=("iqr", "max"),
    )
    .reset_index()
)

within_lineage_dispersion

,system,consensus_program_id,eligible_groups,median_sd,minimum_sd,maximum_sd,median_iqr,minimum_iqr,maximum_iqr
0,cell_line,CONSENSUS_TX_01,16,0.206519,0.142552,0.730968,0.232665,0.157828,1.317082
1,cell_line,CONSENSUS_TX_02,16,0.565270,0.148730,1.131305,0.587655,0.199745,2.180061
2,cell_line,CONSENSUS_TX_03,16,0.615698,0.187219,1.052312,0.674189,0.151974,1.397454
3,tumor,CONSENSUS_TX_01,33,0.700865,0.395350,1.182046,0.900377,0.508702,1.817703
4,tumor,CONSENSUS_TX_02,33,0.321567,0.167238,0.839171,0.438961,0.190486,1.245640
5,tumor,CONSENSUS_TX_03,33,0.683398,0.269842,0.987267,0.877219,0.318379,1.437890


In [15]:
# =============================================================================
# Define lineage-specific native-score fidelity helper
# =============================================================================

def compute_lineage_fidelity(
    data,
    consensus_column,
    native_column,
    group_column,
    eligible_groups,
):
    """Quantify consensus-to-native score fidelity within eligible lineages."""

    results = []

    for group in eligible_groups:
        subset = data.loc[
            data[group_column].eq(group),
            [consensus_column, native_column],
        ]

        pearson_r = pearsonr(
            subset[consensus_column],
            subset[native_column],
        ).statistic

        spearman_r = spearmanr(
            subset[consensus_column],
            subset[native_column],
        ).statistic

        results.append(
            {
                "lineage_group": group,
                "n": len(subset),
                "pearson_r": pearson_r,
                "spearman_r": spearman_r,
            }
        )

    return pd.DataFrame(results)

In [16]:
# =============================================================================
# Align cell-line native scores to frozen consensus orientation
# =============================================================================

for row in consensus_programs.itertuples(index=False):
    oriented_column = f"{row.cell_line_program}_oriented"

    # ICA signs are arbitrary; notebook 400 froze the multiplier that aligns
    # each cell-line program to the tumor-anchored correspondence orientation.
    cell_line_analysis[oriented_column] = (
        cell_line_analysis[row.cell_line_program]
        * row.orientation_multiplier
    )

In [17]:
# =============================================================================
# Compute lineage-specific native-score fidelity
# =============================================================================

lineage_fidelity_results = []

for row in consensus_programs.itertuples(index=False):
    tumor_fidelity = compute_lineage_fidelity(
        tumor_analysis,
        row.consensus_program_id,
        row.tumor_rna_axis,
        "project_id",
        eligible_tumor_projects,
    ).assign(
        system="tumor",
        consensus_program_id=row.consensus_program_id,
    )

    cell_line_fidelity = compute_lineage_fidelity(
        cell_line_analysis,
        row.consensus_program_id,
        f"{row.cell_line_program}_oriented",
        "OncotreeLineage",
        eligible_cell_line_lineages,
    ).assign(
        system="cell_line",
        consensus_program_id=row.consensus_program_id,
    )

    lineage_fidelity_results.extend(
        [tumor_fidelity, cell_line_fidelity]
    )

lineage_fidelity = pd.concat(
    lineage_fidelity_results,
    ignore_index=True,
)

lineage_fidelity

,lineage_group,n,pearson_r,spearman_r,system,consensus_program_id
0,TCGA-BRCA,1089,0.292367,0.258591,tumor,CONSENSUS_TX_01
1,TCGA-UCEC,543,0.445764,0.365781,tumor,CONSENSUS_TX_01
2,TCGA-HNSC,520,0.301440,0.280007,tumor,CONSENSUS_TX_01
3,TCGA-LGG,513,0.218469,0.203380,tumor,CONSENSUS_TX_01
4,TCGA-LUAD,512,0.360293,0.337083,tumor,CONSENSUS_TX_01
...,...,...,...,...,...,...
142,Kidney,19,0.221913,0.152632,cell_line,CONSENSUS_TX_03
143,Peripheral Nervous System,18,0.703353,0.706914,cell_line,CONSENSUS_TX_03
144,Bone,17,0.391779,0.470588,cell_line,CONSENSUS_TX_03
145,Bladder/Urinary Tract,16,-0.087803,-0.261765,cell_line,CONSENSUS_TX_03


In [18]:
# =============================================================================
# Summarize lineage-specific native-score fidelity
# =============================================================================

lineage_fidelity_summary = (
    lineage_fidelity
    .groupby(
        [
            "system",
            "consensus_program_id",
        ]
    )
    .agg(
        eligible_groups=("lineage_group", "size"),
        median_pearson=("pearson_r", "median"),
        minimum_pearson=("pearson_r", "min"),
        maximum_pearson=("pearson_r", "max"),
        positive_pearson_fraction=(
            "pearson_r",
            lambda x: (x > 0).mean(),
        ),
        median_spearman=("spearman_r", "median"),
        minimum_spearman=("spearman_r", "min"),
        maximum_spearman=("spearman_r", "max"),
        positive_spearman_fraction=(
            "spearman_r",
            lambda x: (x > 0).mean(),
        ),
    )
    .reset_index()
)

lineage_fidelity_summary

,system,consensus_program_id,eligible_groups,median_pearson,minimum_pearson,maximum_pearson,positive_pearson_fraction,median_spearman,minimum_spearman,maximum_spearman,positive_spearman_fraction
0,cell_line,CONSENSUS_TX_01,16,0.264308,-0.056828,0.825063,0.937500,0.281618,0.021994,0.793858,1.000000
1,cell_line,CONSENSUS_TX_02,16,0.439091,0.105687,0.933603,1.000000,0.461289,-0.087719,0.721709,0.937500
2,cell_line,CONSENSUS_TX_03,16,0.380220,-0.087803,0.794270,0.875000,0.275093,-0.261765,0.771693,0.875000
3,tumor,CONSENSUS_TX_01,33,0.315297,-0.076815,0.644216,0.969697,0.281198,-0.137037,0.525753,0.848485
4,tumor,CONSENSUS_TX_02,33,0.272270,0.025873,0.651185,1.000000,0.249649,0.006446,0.541737,1.000000
5,tumor,CONSENSUS_TX_03,33,0.157547,-0.115895,0.662906,0.909091,0.146844,-0.130721,0.643372,0.848485


In [19]:
# =============================================================================
# Define leave-one-lineage-out fidelity helper
# =============================================================================

def compute_leave_one_lineage_out_fidelity(
    data,
    consensus_column,
    native_column,
    group_column,
):
    """Measure global consensus-to-native fidelity after excluding each lineage."""

    full_pearson = pearsonr(
        data[consensus_column],
        data[native_column],
    ).statistic

    full_spearman = spearmanr(
        data[consensus_column],
        data[native_column],
    ).statistic

    results = []

    for group in data[group_column].drop_duplicates():
        subset = data.loc[data[group_column].ne(group)]

        results.append(
            {
                "excluded_lineage": group,
                "excluded_n": int(data[group_column].eq(group).sum()),
                "pearson_r": pearsonr(
                    subset[consensus_column],
                    subset[native_column],
                ).statistic,
                "spearman_r": spearmanr(
                    subset[consensus_column],
                    subset[native_column],
                ).statistic,
                "full_pearson_r": full_pearson,
                "full_spearman_r": full_spearman,
            }
        )

    return pd.DataFrame(results)

In [20]:
# =============================================================================
# Compute leave-one-lineage-out native-score fidelity
# =============================================================================

leave_one_lineage_out_results = []

for row in consensus_programs.itertuples(index=False):
    tumor_lolo = compute_leave_one_lineage_out_fidelity(
        tumor_analysis,
        row.consensus_program_id,
        row.tumor_rna_axis,
        "project_id",
    ).assign(
        system="tumor",
        consensus_program_id=row.consensus_program_id,
    )

    cell_line_lolo = compute_leave_one_lineage_out_fidelity(
        cell_line_analysis,
        row.consensus_program_id,
        f"{row.cell_line_program}_oriented",
        "OncotreeLineage",
    ).assign(
        system="cell_line",
        consensus_program_id=row.consensus_program_id,
    )

    leave_one_lineage_out_results.extend(
        [tumor_lolo, cell_line_lolo]
    )

leave_one_lineage_out_fidelity = pd.concat(
    leave_one_lineage_out_results,
    ignore_index=True,
)

leave_one_lineage_out_fidelity

,excluded_lineage,excluded_n,pearson_r,spearman_r,full_pearson_r,full_spearman_r,system,consensus_program_id
0,TCGA-GBM,230,0.262979,0.271366,0.262281,0.269140,tumor,CONSENSUS_TX_01
1,TCGA-OV,422,0.256811,0.263678,0.262281,0.269140,tumor,CONSENSUS_TX_01
2,TCGA-LUAD,512,0.262721,0.272527,0.262281,0.269140,tumor,CONSENSUS_TX_01
3,TCGA-LUSC,500,0.260723,0.265736,0.262281,0.269140,tumor,CONSENSUS_TX_01
4,TCGA-PRAD,496,0.265329,0.278328,0.262281,0.269140,tumor,CONSENSUS_TX_01
...,...,...,...,...,...,...,...,...
175,Pleura,7,0.356060,0.211254,0.366253,0.222374,cell_line,CONSENSUS_TX_03
176,Bone,17,0.360878,0.215480,0.366253,0.222374,cell_line,CONSENSUS_TX_03
177,Prostate,7,0.370334,0.227282,0.366253,0.222374,cell_line,CONSENSUS_TX_03
178,Testis,2,0.365329,0.220280,0.366253,0.222374,cell_line,CONSENSUS_TX_03


In [21]:
# =============================================================================
# Quantify leave-one-lineage-out fidelity shifts
# =============================================================================

leave_one_lineage_out_fidelity["pearson_shift"] = (
    leave_one_lineage_out_fidelity["pearson_r"]
    - leave_one_lineage_out_fidelity["full_pearson_r"]
)

leave_one_lineage_out_fidelity["spearman_shift"] = (
    leave_one_lineage_out_fidelity["spearman_r"]
    - leave_one_lineage_out_fidelity["full_spearman_r"]
)

# Absolute shifts quantify influence irrespective of whether exclusion increases
# or decreases the frozen full-cohort consensus-to-native fidelity.
leave_one_lineage_out_fidelity["abs_pearson_shift"] = (
    leave_one_lineage_out_fidelity["pearson_shift"].abs()
)

leave_one_lineage_out_fidelity["abs_spearman_shift"] = (
    leave_one_lineage_out_fidelity["spearman_shift"].abs()
)

leave_one_lineage_out_fidelity

,excluded_lineage,excluded_n,pearson_r,spearman_r,full_pearson_r,full_spearman_r,system,consensus_program_id,pearson_shift,spearman_shift,abs_pearson_shift,abs_spearman_shift
0,TCGA-GBM,230,0.262979,0.271366,0.262281,0.269140,tumor,CONSENSUS_TX_01,0.000699,0.002226,0.000699,0.002226
1,TCGA-OV,422,0.256811,0.263678,0.262281,0.269140,tumor,CONSENSUS_TX_01,-0.005470,-0.005462,0.005470,0.005462
2,TCGA-LUAD,512,0.262721,0.272527,0.262281,0.269140,tumor,CONSENSUS_TX_01,0.000441,0.003387,0.000441,0.003387
3,TCGA-LUSC,500,0.260723,0.265736,0.262281,0.269140,tumor,CONSENSUS_TX_01,-0.001557,-0.003403,0.001557,0.003403
4,TCGA-PRAD,496,0.265329,0.278328,0.262281,0.269140,tumor,CONSENSUS_TX_01,0.003048,0.009188,0.003048,0.009188
...,...,...,...,...,...,...,...,...,...,...,...,...
175,Pleura,7,0.356060,0.211254,0.366253,0.222374,cell_line,CONSENSUS_TX_03,-0.010193,-0.011120,0.010193,0.011120
176,Bone,17,0.360878,0.215480,0.366253,0.222374,cell_line,CONSENSUS_TX_03,-0.005375,-0.006893,0.005375,0.006893
177,Prostate,7,0.370334,0.227282,0.366253,0.222374,cell_line,CONSENSUS_TX_03,0.004081,0.004908,0.004081,0.004908
178,Testis,2,0.365329,0.220280,0.366253,0.222374,cell_line,CONSENSUS_TX_03,-0.000924,-0.002094,0.000924,0.002094


In [22]:
# =============================================================================
# Summarize leave-one-lineage-out influence
# =============================================================================

leave_one_lineage_out_summary = (
    leave_one_lineage_out_fidelity
    .groupby(
        [
            "system",
            "consensus_program_id",
        ]
    )
    .agg(
        full_pearson_r=("full_pearson_r", "first"),
        maximum_abs_pearson_shift=("abs_pearson_shift", "max"),
        full_spearman_r=("full_spearman_r", "first"),
        maximum_abs_spearman_shift=("abs_spearman_shift", "max"),
    )
    .reset_index()
)

leave_one_lineage_out_summary

,system,consensus_program_id,full_pearson_r,maximum_abs_pearson_shift,full_spearman_r,maximum_abs_spearman_shift
0,cell_line,CONSENSUS_TX_01,0.653557,0.477405,0.367358,0.054145
1,cell_line,CONSENSUS_TX_02,0.591080,0.127141,0.345068,0.076311
2,cell_line,CONSENSUS_TX_03,0.366253,0.023845,0.222374,0.024734
3,tumor,CONSENSUS_TX_01,0.262281,0.031878,0.269140,0.021653
4,tumor,CONSENSUS_TX_02,0.128220,0.013133,0.120032,0.010077
5,tumor,CONSENSUS_TX_03,0.236501,0.059858,0.208687,0.050933


In [23]:
# =============================================================================
# Identify most influential lineage exclusions
# =============================================================================

most_influential_lineages = (
    leave_one_lineage_out_fidelity
    .loc[
        leave_one_lineage_out_fidelity
        .groupby(
            [
                "system",
                "consensus_program_id",
            ]
        )["abs_pearson_shift"]
        .idxmax(),
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
            "excluded_n",
            "full_pearson_r",
            "pearson_r",
            "pearson_shift",
            "full_spearman_r",
            "spearman_r",
            "spearman_shift",
        ],
    ]
    .reset_index(drop=True)
)

most_influential_lineages

,system,consensus_program_id,excluded_lineage,excluded_n,full_pearson_r,pearson_r,pearson_shift,full_spearman_r,spearman_r,spearman_shift
0,cell_line,CONSENSUS_TX_01,Lymphoid,86,0.653557,0.176152,-0.477405,0.367358,0.313213,-0.054145
1,cell_line,CONSENSUS_TX_02,Head and Neck,26,0.591080,0.463939,-0.127141,0.345068,0.268757,-0.076311
2,cell_line,CONSENSUS_TX_03,Breast,47,0.366253,0.342408,-0.023845,0.222374,0.211285,-0.011089
3,tumor,CONSENSUS_TX_01,TCGA-LAML,134,0.262281,0.294159,0.031878,0.269140,0.275542,0.006402
4,tumor,CONSENSUS_TX_02,TCGA-LGG,513,0.128220,0.141352,0.013133,0.120032,0.128132,0.008101
5,tumor,CONSENSUS_TX_03,TCGA-BRCA,1089,0.236501,0.176643,-0.059858,0.208687,0.157754,-0.050933


In [24]:
# =============================================================================
# Identify most influential lineage exclusions for Spearman fidelity
# =============================================================================

most_influential_spearman_lineages = (
    leave_one_lineage_out_fidelity
    .loc[
        leave_one_lineage_out_fidelity
        .groupby(
            [
                "system",
                "consensus_program_id",
            ]
        )["abs_spearman_shift"]
        .idxmax(),
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
            "excluded_n",
            "full_spearman_r",
            "spearman_r",
            "spearman_shift",
        ],
    ]
    .reset_index(drop=True)
)

most_influential_spearman_lineages

,system,consensus_program_id,excluded_lineage,excluded_n,full_spearman_r,spearman_r,spearman_shift
0,cell_line,CONSENSUS_TX_01,Lymphoid,86,0.367358,0.313213,-0.054145
1,cell_line,CONSENSUS_TX_02,Head and Neck,26,0.345068,0.268757,-0.076311
2,cell_line,CONSENSUS_TX_03,Lung,136,0.222374,0.247108,0.024734
3,tumor,CONSENSUS_TX_01,TCGA-THCA,505,0.269140,0.247487,-0.021653
4,tumor,CONSENSUS_TX_02,TCGA-UCEC,543,0.120032,0.109955,-0.010077
5,tumor,CONSENSUS_TX_03,TCGA-BRCA,1089,0.208687,0.157754,-0.050933


In [25]:
# =============================================================================
# Integrate cross-lineage robustness evidence
# =============================================================================

# Preserve each robustness dimension separately rather than collapsing
# heterogeneous evidence into a post hoc composite score.
cross_lineage_robustness_summary = (
    lineage_eta_squared
    .merge(
        within_lineage_dispersion,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        lineage_fidelity_summary.drop(columns="eligible_groups"),
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        leave_one_lineage_out_summary,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_robustness_summary

,system,consensus_program_id,lineage_eta_squared,eligible_groups,median_sd,minimum_sd,maximum_sd,median_iqr,minimum_iqr,maximum_iqr,...,maximum_pearson,positive_pearson_fraction,median_spearman,minimum_spearman,maximum_spearman,positive_spearman_fraction,full_pearson_r,maximum_abs_pearson_shift,full_spearman_r,maximum_abs_spearman_shift
0,tumor,CONSENSUS_TX_01,0.458909,33,0.700865,0.395350,1.182046,0.900377,0.508702,1.817703,...,0.644216,0.969697,0.281198,-0.137037,0.525753,0.848485,0.262281,0.031878,0.269140,0.021653
1,cell_line,CONSENSUS_TX_01,0.894808,16,0.206519,0.142552,0.730968,0.232665,0.157828,1.317082,...,0.825063,0.937500,0.281618,0.021994,0.793858,1.000000,0.653557,0.477405,0.367358,0.054145
2,tumor,CONSENSUS_TX_02,0.809194,33,0.321567,0.167238,0.839171,0.438961,0.190486,1.245640,...,0.651185,1.000000,0.249649,0.006446,0.541737,1.000000,0.128220,0.013133,0.120032,0.010077
3,cell_line,CONSENSUS_TX_02,0.565508,16,0.565270,0.148730,1.131305,0.587655,0.199745,2.180061,...,0.933603,1.000000,0.461289,-0.087719,0.721709,0.937500,0.591080,0.127141,0.345068,0.076311
4,tumor,CONSENSUS_TX_03,0.521870,33,0.683398,0.269842,0.987267,0.877219,0.318379,1.437890,...,0.662906,0.909091,0.146844,-0.130721,0.643372,0.848485,0.236501,0.059858,0.208687,0.050933
5,cell_line,CONSENSUS_TX_03,0.612890,16,0.615698,0.187219,1.052312,0.674189,0.151974,1.397454,...,0.794270,0.875000,0.275093,-0.261765,0.771693,0.875000,0.366253,0.023845,0.222374,0.024734


In [26]:
# =============================================================================
# Attach most influential lineage identities
# =============================================================================

pearson_influence = (
    most_influential_lineages[
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
        ]
    ]
    .rename(
        columns={
            "excluded_lineage": "max_pearson_influence_lineage",
        }
    )
)

spearman_influence = (
    most_influential_spearman_lineages[
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
        ]
    ]
    .rename(
        columns={
            "excluded_lineage": "max_spearman_influence_lineage",
        }
    )
)

cross_lineage_robustness_summary = (
    cross_lineage_robustness_summary
    .merge(
        pearson_influence,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        spearman_influence,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_robustness_summary[
    [
        "system",
        "consensus_program_id",
        "max_pearson_influence_lineage",
        "maximum_abs_pearson_shift",
        "max_spearman_influence_lineage",
        "maximum_abs_spearman_shift",
    ]
]

,system,consensus_program_id,max_pearson_influence_lineage,maximum_abs_pearson_shift,max_spearman_influence_lineage,maximum_abs_spearman_shift
0,tumor,CONSENSUS_TX_01,TCGA-LAML,0.031878,TCGA-THCA,0.021653
1,cell_line,CONSENSUS_TX_01,Lymphoid,0.477405,Lymphoid,0.054145
2,tumor,CONSENSUS_TX_02,TCGA-LGG,0.013133,TCGA-UCEC,0.010077
3,cell_line,CONSENSUS_TX_02,Head and Neck,0.127141,Head and Neck,0.076311
4,tumor,CONSENSUS_TX_03,TCGA-BRCA,0.059858,TCGA-BRCA,0.050933
5,cell_line,CONSENSUS_TX_03,Breast,0.023845,Lung,0.024734


In [27]:
# =============================================================================
# Contextualize maximum Pearson influence with within-lineage fidelity
# =============================================================================

# This post hoc diagnostic contextualizes the lineage identified by LOLO.
# It does not redefine robustness criteria or alter consensus classification.
influential_lineage_fidelity = (
    most_influential_lineages[
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
            "excluded_n",
            "pearson_shift",
            "spearman_shift",
        ]
    ]
    .rename(
        columns={
            "excluded_lineage": "lineage_group",
            "excluded_n": "lineage_n",
        }
    )
    .merge(
        lineage_fidelity[
            [
                "system",
                "consensus_program_id",
                "lineage_group",
                "pearson_r",
                "spearman_r",
            ]
        ],
        on=[
            "system",
            "consensus_program_id",
            "lineage_group",
        ],
        how="left",
        validate="one_to_one",
    )
)

influential_lineage_fidelity

,system,consensus_program_id,lineage_group,lineage_n,pearson_shift,spearman_shift,pearson_r,spearman_r
0,cell_line,CONSENSUS_TX_01,Lymphoid,86,-0.477405,-0.054145,0.825063,0.793858
1,cell_line,CONSENSUS_TX_02,Head and Neck,26,-0.127141,-0.076311,0.804699,0.721709
2,cell_line,CONSENSUS_TX_03,Breast,47,-0.023845,-0.011089,0.645815,0.445421
3,tumor,CONSENSUS_TX_01,TCGA-LAML,134,0.031878,0.006402,-0.076815,-0.137037
4,tumor,CONSENSUS_TX_02,TCGA-LGG,513,0.013133,0.008101,0.134683,0.152014
5,tumor,CONSENSUS_TX_03,TCGA-BRCA,1089,-0.059858,-0.050933,0.655532,0.643372


In [28]:
# =============================================================================
# Define cross-lineage robustness output path
# =============================================================================

ROBUSTNESS_SUMMARY_PATH = (
    OUTPUT_DIR
    / "402_cross_lineage_robustness_summary.csv"
)

In [29]:
# =============================================================================
# Write cross-lineage robustness summary
# =============================================================================

cross_lineage_robustness_summary.to_csv(
    ROBUSTNESS_SUMMARY_PATH,
    index=False,
)

In [30]:
# =============================================================================
# Verify cross-lineage robustness artifact publication
# =============================================================================

published_artifacts = [
    ROBUSTNESS_SUMMARY_PATH,
]

print(
    "Cross-lineage robustness artifacts published:",
    all(path.exists() for path in published_artifacts),
)
print("Artifacts written:", len(published_artifacts))

Cross-lineage robustness artifacts published: True
Artifacts written: 1


## Cross-lineage robustness summary

The three frozen consensus transcriptomic representations showed measurable
lineage structure in both tumor and cell-line systems, with heterogeneous
within-lineage variation and native-program fidelity across contexts.

`CONSENSUS_TX_01` showed lineage eta-squared values of 0.459 in tumors and
0.895 in cell lines. Median within-lineage SD was 0.701 in tumors and 0.207
in cell lines. Median lineage-specific Pearson fidelity was 0.315 in tumors
and 0.264 in cell lines. In cell lines, exclusion of the Lymphoid lineage
reduced global Pearson fidelity from 0.654 to 0.176, while Lymphoid itself
showed Pearson = 0.825 and Spearman = 0.794. This identifies a specific
lineage context with substantial influence on the global linear association.

`CONSENSUS_TX_02` showed lineage eta-squared values of 0.809 in tumors and
0.566 in cell lines. Median within-lineage SD was 0.322 and 0.565,
respectively. Pearson fidelity was positive in all eligible tumor projects
and all eligible cell-line lineages. The largest leave-one-lineage-out
Pearson shift occurred after exclusion of Head and Neck cell lines
(|Δr| = 0.127).

`CONSENSUS_TX_03` showed lineage eta-squared values of 0.522 in tumors and
0.613 in cell lines, with median within-lineage SD values of 0.683 and 0.616.
Median lineage-specific Pearson fidelity was 0.158 in tumors and 0.380 in
cell lines. The largest tumor-side leave-one-lineage-out Pearson shift was
observed after exclusion of TCGA-BRCA (|Δr| = 0.060), whereas the maximum
cell-line shift was 0.024.

These results indicate that the frozen consensus representations are not
lineage-independent and that their cross-lineage portability differs across
programs and systems. Lineage-specific correlation estimates were restricted
to prespecified groups with n >= 15, while leave-one-lineage-out analyses
evaluated all groups without refitting or reweighting the consensus
representations.

The contextual inspection of the most influential lineage exclusions was
performed post hoc for interpretation only and did not alter the frozen
analysis criteria, consensus definitions, weights, orientations, or
classification.

Notebook 402 provides internal cross-lineage robustness evidence for the
frozen Phase 4 consensus layer. It does not establish independent validation,
biological causality, clinical prediction, or full
epigenetic-transcriptomic reproduction.